# Mathematical machines with Flax NNX

Flax helps construct parametrised machines from elementary layers. We use its NNX interface: a machine is a Python object that can be evaluated, differentiated and fitted.

The aim of this notebook is simple: **define a mathematical machine in a few lines, then learn its parameters from data**.

By the end of this notebook, you should be able to:

- define a multilayer perceptron (MLP) with `nnx.Module`;
- relate its layers to the mathematical expression of the machine;
- identify its parameters $p$;
- perform learning with `nnx.value_and_grad`, Optax and `nnx.Optimizer`.

## Exercises

1. [An affine layer](#Exercise-1)
2. [The machine class matters](#Exercise-2)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import flax
from flax import nnx

print(f"JAX {jax.__version__}, Flax {flax.__version__}, Optax {optax.__version__}")

## 1. Defining a machine

Consider a machine $M_p:\mathbb R\to\mathbb R$ of the form

$$
M_p(x)=W_2\tanh(W_1x+b_1)+b_2.
$$

The parameter $p$ collects the matrices and vectors $(W_1,b_1,W_2,b_2)$. The dimension of the intermediate space is called `largeur` in the program.

In [ ]:
class PerceptronMulticouche(nnx.Module):
    def __init__(self, largeur, *, rngs):
        self.affine_1 = nnx.Linear(1, largeur, rngs=rngs)
        self.affine_2 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        return self.affine_2(jnp.tanh(self.affine_1(x)))

A class derived from `nnx.Module` describes a module that can be composed with other modules. Its `__call__` method describes the evaluation of the machine (the forward pass).

Despite its name, `nnx.Linear` represents an **affine** map by default, with a `kernel` matrix and a `bias` vector. NNX uses row vectors and computes `x @ kernel + bias`. With the course convention of column vectors, we therefore have $W=\mathtt{kernel}^\top$.

The object `nnx.Rngs(0)` provides a reproducible sequence of keys for random parameter initialisation.

In [ ]:
machine = PerceptronMulticouche(largeur=32, rngs=nnx.Rngs(0))
x_test = jnp.array([[-1.0], [0.0], [1.0]])

print(f"shape of x: {x_test.shape}")
print(f"shape of M_p(x): {machine(x_test).shape}")
print(f"W_1 : {machine.affine_1.kernel[...].T.shape}")
print(f"b_1 : {machine.affine_1.bias[...].shape}")
print(f"W_2 : {machine.affine_2.kernel[...].T.shape}")
print(f"b_2 : {machine.affine_2.bias[...].shape}")

### Exercise 1

**An affine layer.**

Construct `affine = nnx.Linear(2, 3, rngs=nnx.Rngs(1))` and choose an array `x` of shape `(4, 2)`. Verify directly that

```python
affine(x) == x @ affine.kernel[...] + affine.bias[...]
```

Explain the shapes of all arrays and give the dimensions of the corresponding affine map.

In [ ]:
# Complete this section.

## 2. Data and objective function

We observe the function

$$
f(x)=\sin(2\pi x)+0.3x
$$

at $n_{\mathcal D}=129$ points. The arrays have shape `(129, 1)`: the first axis runs through the data and the second represents the input or output dimension.

In [ ]:
def cible(x):
    return jnp.sin(2.0 * jnp.pi * x) + 0.3 * x

x_observe = jnp.linspace(-1.0, 1.0, 129).reshape(-1, 1)
z_observe = cible(x_observe)

The objective function is the mean squared error. It depends on all parameters $p$ contained in `machine`, even though we do not explicitly collect them in a vector.

In [ ]:
def cout(machine, x, z):
    residu = machine(x) - z
    return 0.5 * jnp.mean(residu**2)

print(f"initial objective: {cout(machine, x_observe, z_observe):.3e}")

## 3. Learning the parameters

The matrices and biases created by `nnx.Linear` are variables of type `nnx.Param`. The argument `wrt=nnx.Param` specifies that these are the variables the optimiser should fit.

The `nnx.value_and_grad` transformation computes the objective and its gradient with respect to these parameters simultaneously. `nnx.Optimizer` associates the Optax transformation with its internal state. Its `update` method modifies the machine and optimiser state **in place**: this is where the mutability of Python objects, introduced in notebook 00, comes into play.

Finally, `@nnx.jit` plays the same role as `@jax.jit`, while propagating changes to NNX objects outside the compiled function.

In [ ]:
alpha = 1.0e-2
optimiseur = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)

@nnx.jit
def pas_apprentissage(machine, optimiseur, x, z):
    valeur, gradient = nnx.value_and_grad(cout)(machine, x, z)
    optimiseur.update(machine, gradient)
    return valeur

The training loop contains no network-specific computation: it simply repeats the preceding step.

In [ ]:
historique = []
for _ in range(1500):
    valeur = pas_apprentissage(machine, optimiseur, x_observe, z_observe)
    historique.append(float(valeur))

print(f"final objective: {cout(machine, x_observe, z_observe):.3e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(x_observe[:, 0], z_observe[:, 0], label="data")
axes[0].plot(x_observe[:, 0], machine(x_observe)[:, 0], "--", label="machine")
axes[0].set_xlabel("x")
axes[0].legend()

axes[1].semilogy(historique)
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("objective function")

fig.tight_layout()

Distinguish the roles carefully: Flax organises the machine and its parameters, JAX computes the gradient and compiles the step, while Optax transforms the gradient into an update. NNX connects these objects without requiring us to manipulate the parameter tree ourselves.

### Exercise 2

**The machine class matters.**

Define a class `MachineAffine` containing a single `nnx.Linear(1, 1)` layer. Fit it to the same data with the same learning rate and compare its final objective with that of the multilayer perceptron.

Why does extending training fail to remove the remaining discrepancy? Is this a limitation of the optimisation method or of the machine class?

In [ ]:
# Complete this section.

## Summary

The Python class defines the machine class $\{M_p\}$, calling `machine(x)` evaluates $M_p(x)$, and `nnx.value_and_grad` computes the gradient of the objective with respect to $p$. With `nnx.Optimizer`, only a few lines are then needed to learn these parameters.

The simplicity of the program does not remove the mathematical questions: the choice of machine class, data quality, objective function and optimisation behaviour. Instead, it allows us to investigate them directly.